In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/val.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/train.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/test.csv
/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0/dataset.csv


In [2]:
!pip install transformers datasets torch scikit-learn underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.7 MB/s eta 0:00:00


In [3]:
import pandas as pd

BASE = "/kaggle/input/datasets/natsumekafka/vietnamese-ai-detection-v0"
train_df = pd.read_csv(f"{BASE}/train.csv")
val_df   = pd.read_csv(f"{BASE}/val.csv")
test_df  = pd.read_csv(f"{BASE}/test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["label"].value_counts())

(2993, 9) (374, 9) (375, 9)
label
ai       1497
human    1496
Name: count, dtype: int64


In [4]:
from underthesea import word_tokenize
from tqdm import tqdm
tqdm.pandas()

def segment(text):
    return word_tokenize(str(text), format="text")

print("Segmenting train...")
train_df["text_seg"] = train_df["text"].progress_apply(segment)

print("Segmenting val...")
val_df["text_seg"]   = val_df["text"].progress_apply(segment)

print("Segmenting test...")
test_df["text_seg"]  = test_df["text"].progress_apply(segment)

# Lưu cache tránh chạy lại
train_df.to_csv("/kaggle/working/train_seg.csv", index=False)
val_df.to_csv("/kaggle/working/val_seg.csv",   index=False)
test_df.to_csv("/kaggle/working/test_seg.csv",  index=False)
print("Done! Ví dụ:", train_df["text_seg"].iloc[0][:80])

Segmenting train...


100%|██████████| 2993/2993 [01:16<00:00, 39.17it/s]


Segmenting val...


100%|██████████| 374/374 [00:09<00:00, 39.01it/s]


Segmenting test...


100%|██████████| 375/375 [00:08<00:00, 42.67it/s]


Done! Ví dụ: Xã_hội ngày_càng phát_triển , con_người càng phải đối_mặt với những vấn_đề tiêu_


In [5]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class ViDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(df["text_seg"]),   # ← dùng text đã segment
            truncation=True,
            padding=True,
            max_length=max_len       # PhoBERT max = 256
        )
        self.labels = list(df["label_int"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_ds = ViDataset(train_df, tokenizer)
val_ds   = ViDataset(val_df,   tokenizer)
test_ds  = ViDataset(test_df,  tokenizer)
print("Datasets ready:", len(train_ds), len(val_ds), len(test_ds))

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Datasets ready: 2993 374 375


In [6]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer)
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="binary")
    }

args = TrainingArguments(
    output_dir="./phobert-ai-detection",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)
trainer.train()

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=470, training_loss=0.25963413151813314, metrics={'train_runtime': 453.0119, 'train_samples_per_second': 33.034, 'train_steps_per_second': 1.038, 'total_flos': 1968728471731200.0, 'train_loss': 0.25963413151813314, 'epoch': 5.0})

In [7]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predict trên test set
pred_output = trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

# Metrics tổng quan
print("=== Test Metrics ===")
print(pred_output.metrics)

# Chi tiết từng class
print("\n=== Classification Report ===")
print(classification_report(labels, preds, target_names=["human", "ai"]))

# Confusion matrix
print("=== Confusion Matrix ===")
print(confusion_matrix(labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


=== Test Metrics ===
{'test_loss': 0.266105055809021, 'test_accuracy': 0.976, 'test_f1': 0.9765013054830287, 'test_runtime': 3.4803, 'test_samples_per_second': 107.748, 'test_steps_per_second': 1.724}

=== Classification Report ===
              precision    recall  f1-score   support

       human       1.00      0.95      0.98       188
          ai       0.95      1.00      0.98       187

    accuracy                           0.98       375
   macro avg       0.98      0.98      0.98       375
weighted avg       0.98      0.98      0.98       375

=== Confusion Matrix ===
[[179   9]
 [  0 187]]


In [8]:
trainer.save_model("/kaggle/working/phobert-finetuned")
tokenizer.save_pretrained("/kaggle/working/phobert-finetuned")
print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [9]:
import os
files = os.listdir("/kaggle/working/phobert-finetuned")
print(files)

['bpe.codes', 'tokenizer_config.json', 'config.json', 'added_tokens.json', 'vocab.txt', 'model.safetensors', 'training_args.bin']
